# Downstream forecasting utility

Does repairing a damaged history help a later forecast? A small Ridge forecaster is fitted once on clean data and then held fixed while its input history changes.

**Input:** completed results in `artifacts/evaluation/corrected_v2_downstream_v2/`. This notebook reads saved predictions and summaries.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

## Load one variable

Set `SELECTED_VARIABLE` to temperature, humidity, pressure, precipitation or wind direction. Wind speed has no eligible station under this split and is explicitly not evaluable.
Each evaluable variable represents **one selected station**, with five seeds, four missingness cases and four forecast horizons. Missing/incomplete artifacts raise an error.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from notebooks.release_artifacts import load_downstream  # noqa: E402

SELECTED_VARIABLE = "temperature"
payload, artifact_path = load_downstream(PROJECT_ROOT, SELECTED_VARIABLE)
records = pd.DataFrame(payload["records"])
summaries = pd.DataFrame(payload["summaries"])
artifact_path, payload["protocol"]

## Mean utility across seeds

`mae` is forecast error in the selected variable's units; lower is better. `delta_vs_corrupted` compares each method with the neutral-filled damaged history, so negative is an improvement.
`seed_sd` describes variation across the five paired seeds. Recovery ratios are unstable when clean and corrupted errors are nearly equal.


In [ ]:
group_columns = [
    "variable",
    "station",
    "mask_type",
    "mask_parameter",
    "method",
    "horizon",
]
mean_summary = summaries.groupby(group_columns, as_index=False).agg(
    mae=("mae", "mean"),
    rmse=("rmse", "mean"),
    recovery_ratio=("recovery_ratio", "mean"),
    delta_vs_corrupted=("mae_difference_vs_corrupted", "mean"),
    seed_sd=("mae", "std"),
)
mean_summary.sort_values(
    ["variable", "mask_type", "mask_parameter", "horizon", "mae"]
).head(40)

## Forecast MAE by horizon

The default plot uses the terminal-gap case: missing observations immediately before forecasting. Change `selected_mask` to inspect another case.


In [ ]:
selected_variable = mean_summary["variable"].iloc[0]
selected_mask = (
    "terminal"
    if "terminal" in set(mean_summary["mask_type"])
    else mean_summary["mask_type"].iloc[0]
)
view = mean_summary.query(
    "variable == @selected_variable and mask_type == @selected_mask"
)
parameter = view["mask_parameter"].max()
view = view.query("mask_parameter == @parameter")

fig, ax = plt.subplots(figsize=(9, 5))
for method, group in view.groupby("method"):
    group = group.sort_values("horizon")
    ax.plot(group["horizon"], group["mae"], marker="o", label=method)
ax.set(
    title=f"{selected_variable}: {selected_mask} {parameter}",
    xlabel="Forecast horizon (h)",
    ylabel="Forecast MAE",
)
ax.legend(ncol=2)
ax.grid(alpha=0.25)
plt.show()

## Failures remain visible

The table below lists cases where a repaired history forecasts worse than the neutral corrupted history.

In [ ]:
failures = mean_summary.query(
    "method in ['interpolation', 'bilstm_fair', 'giano'] and delta_vs_corrupted > 0"
).sort_values("delta_vs_corrupted", ascending=False)
failures.head(30)